# KOSPI FinRL + JSONL 임베딩 파이프라인

## 실제 임베딩 스키마
```json
{"row_id": 0, "source_type": "market", "date": "2026.05.21",
 "company": "시장전체", "ticker": "", "keyword": "중국 경기",
 "title": "미 국채 금리 상승에 한중일 통화 비상[천조국 리포트]",
 "press": "머니투데이방송", "source": "naver",
 "embedding_model": "upstage-autoencoder-16d", "embedding_dim": 16,
 "embedding": [-0.363, 0.019, 0.753, ...]}
```
- `source_type = "market"`  → 시장 전체 뉴스 (날짜별 broadcast)
- `source_type = "company"` → 종목별 뉴스 (ticker 기준 join)

In [ ]:
# 의존성 설치 (최초 1회)
!pip install yfinance stockstats stable-baselines3[extra] gymnasium \
             google-api-python-client google-auth-oauthlib \
             scikit-learn pandas numpy matplotlib pyarrow -q

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

# ── 설정 ────────────────────────────────────────────────────────
TICKERS = ['005930.KS', '000660.KS', '035420.KS', '005380.KS', '051910.KS']
TRAIN_START       = '2025-01-01'
TRAIN_END         = '2026-01-01'
TEST_END          = '2026-05-30'
EMB_DIM           = 16          # Upstage autoencoder-16d
ALGORITHM         = 'ppo'
TIMESTEPS         = 50_000
INITIAL_KRW       = 10_000_000

# 임베딩 소스 설정 (둘 중 하나)
GDRIVE_FOLDER_ID  = None        # 예: '1BxiMVs0XRA5nFMdKvBdBZjgmUUqptlbs'
LOCAL_JSONL_PATH  = None        # 예: './embeddings/' 또는 './data.jsonl'

print('설정 완료')

## Step 1 — KOSPI 시장 데이터 (Yahoo Finance)

In [ ]:
from data.market_data import KOSPIDataLoader

loader = KOSPIDataLoader(tickers=TICKERS, start=TRAIN_START, end=TEST_END)
df_market = loader.fetch_and_process()
print(f'행: {len(df_market):,} | 종목: {df_market["tic"].nunique()}')
df_market.head(3)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(TICKERS), 1, figsize=(13, 2.5*len(TICKERS)))
for ax, tic in zip(axes, TICKERS):
    s = df_market[df_market['tic']==tic].sort_values('date')
    ax.plot(s['date'].values, s['close'].values, lw=0.9)
    ax.set_title(tic, fontsize=9); ax.set_xticks([])
plt.suptitle('KOSPI 종가', y=1.01); plt.tight_layout(); plt.show()

## Step 2 — JSONL 임베딩 로드

In [ ]:
import pandas as pd
from data.gdrive_loader import (
    GDriveEmbeddingLoader, LocalEmbeddingLoader, make_dummy_embeddings
)

if GDRIVE_FOLDER_ID:
    # ── Google Drive에서 .jsonl 파일 다운로드 ─────────────────
    emb_loader = GDriveEmbeddingLoader(folder_id=GDRIVE_FOLDER_ID, emb_dim=EMB_DIM)
    df_emb = emb_loader.load()

elif LOCAL_JSONL_PATH:
    # ── 로컬 .jsonl 파일 로드 ────────────────────────────────
    emb_loader = LocalEmbeddingLoader(path=LOCAL_JSONL_PATH, emb_dim=EMB_DIM)
    df_emb = emb_loader.load()

else:
    # ── 더미 임베딩 (테스트용) ───────────────────────────────
    print('⚠️  임베딩 소스 미설정 → 더미 임베딩 사용 (dim=16)')
    dates = sorted(df_market['date'].unique())
    df_emb = make_dummy_embeddings(
        dates=dates, tickers=TICKERS,
        emb_dim=EMB_DIM, include_market=True
    )

print(f'임베딩 DataFrame: {df_emb.shape}')
print(f"source_type 분포:\n{df_emb['source_type'].value_counts().to_string()}")
df_emb.head(3)

In [ ]:
# 실제 JSONL 파일을 직접 파싱해보기 (파일 경로를 지정)
import json
SAMPLE_JSONL = None  # './your_file.jsonl'

if SAMPLE_JSONL:
    from data.gdrive_loader import _parse_jsonl_to_df
    from pathlib import Path
    df_sample = _parse_jsonl_to_df(Path(SAMPLE_JSONL), emb_dim=EMB_DIM)
    print(df_sample.head())
    print('임베딩 dim:', len([c for c in df_sample.columns if c.startswith('emb_')]))

## Step 3 — State 구성 (시장 + 임베딩 결합)

In [ ]:
from data.state_builder import StateBuilder

builder = StateBuilder(
    df_market=df_market,
    df_embeddings=df_emb,
    emb_dim=EMB_DIM,
    train_end=TRAIN_END,
    use_company_emb=True,  # source_type="company" 임베딩 사용
    use_market_emb=True,   # source_type="market"  임베딩 사용
)
df_train, df_test = builder.build_train_test()
print(f'학습 행: {len(df_train):,} | 테스트 행: {len(df_test):,}')

In [ ]:
# 환경 sanity check
from rl.env_kospi import KOSPITradingEnv

TECH_INDICATORS = [
    'macd','boll_ub','boll_lb','rsi_30','cci_30','dx_30',
    'close_30_sma','close_60_sma','volume_change','return_1d','return_5d'
]
tech_cols = [c for c in TECH_INDICATORS if c in df_train.columns]

env = KOSPITradingEnv(
    df=df_train, tech_indicators=tech_cols,
    emb_dim=builder.emb_dim, initial_capital=INITIAL_KRW,
)
obs, _ = env.reset()
print(f'obs shape: {obs.shape} (expected: ({builder.state_dim},))')
print(f'action space: {env.action_space}')

act = env.action_space.sample()
obs2, rew, done, _, info = env.step(act)
print(f'reward: {rew:.6f} | 포트폴리오: {info["portfolio_value"]:,.0f} KRW')

## Step 4 — 강화학습 학습

In [ ]:
from rl.trainer import RLTrainer

trainer = RLTrainer(
    df_train=df_train, df_test=df_test,
    state_dim=builder.state_dim, n_stocks=len(TICKERS),
    initial_capital=INITIAL_KRW,
    algo=ALGORITHM, timesteps=TIMESTEPS,
    output_dir='results/', emb_dim=builder.emb_dim,
)
model = trainer.train()

## Step 5 — 평가

In [ ]:
metrics = trainer.evaluate(model)

In [ ]:
from IPython.display import Image
Image(f'results/portfolio_{ALGORITHM}.png')

## 부록 — 실제 JSONL 파일 활용 방법

### 로컬 파일 직접 사용
```python
from data.gdrive_loader import LocalEmbeddingLoader
df_emb = LocalEmbeddingLoader(path='./embeddings/', emb_dim=16).load()
# 또는
df_emb = LocalEmbeddingLoader(path='./data.jsonl', emb_dim=16).load()
```

### Google Drive 서비스 계정 사용
```python
from data.gdrive_loader import GDriveEmbeddingLoader
df_emb = GDriveEmbeddingLoader(
    folder_id='YOUR_FOLDER_ID',
    emb_dim=16,
    credentials_path='service_account.json',
).load()
```

### Google Drive OAuth2 브라우저 플로우
```python
# 프로젝트 루트에 credentials.json 배치 후
df_emb = GDriveEmbeddingLoader(folder_id='YOUR_FOLDER_ID', emb_dim=16).load()
# → 최초 실행 시 브라우저 열림
```

### emb_dim 자동 감지
```python
# emb_dim=None → JSONL에서 embedding_dim 필드 자동 읽기
df_emb = LocalEmbeddingLoader(path='./data.jsonl', emb_dim=None).load()
builder = StateBuilder(df_market, df_emb, emb_dim=None, train_end='2026-01-01')
```